In [7]:
import openai
print(openai.__version__)

1.64.0


In [ ]:
import os
import openai

openai.api_key = os.getenv("OPENAI_API_KEY")


In [9]:
# ================================
# Week 3 – LLM Prompt Design
# ================================

SLA_PROMPT_TEMPLATE = """
You are a legal contract analysis assistant specialized in car lease agreements.

Extract the following SLA details from the contract text.
If a value is missing, return null.
Return ONLY valid JSON. No explanation.

Fields:
- interest_rate_apr
- lease_term_months
- monthly_payment
- down_payment
- residual_value
- mileage_allowance
- overage_charge
- early_termination
- purchase_option
- maintenance_responsibility
- warranty_insurance
- penalties
- missing_or_ambiguous_clauses

Contract Text:
\"\"\"
{contract_text}
\"\"\"
"""

In [10]:
# ================================
# Week 3 – OpenAI LLM Extraction
# ================================



def extract_sla_with_gpt(contract_text):
    prompt = SLA_PROMPT_TEMPLATE.format(contract_text=contract_text)

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    raw_output = response.output_text.strip()

    match = re.search(r"\{[\s\S]*\}", raw_output)
    if not match:
        return {"error": "No JSON found", "raw_output": raw_output}

    try:
        return json.loads(match.group(0))
    except Exception as e:
        return {"error": str(e), "json_text": match.group(0)}

In [13]:
recalls = fetch_vehicle_recalls(
    vehicle_info["make"],
    vehicle_info["model"],
    vehicle_info["year"]
)

recalls[:2]

NameError: name 'fetch_vehicle_recalls' is not defined

In [8]:
import requests

def get_vehicle_details(vin):
    url = f"https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVinValues/{vin}?format=json"
    response = requests.get(url)
    return response.json()["Results"][0]

In [9]:
def extract_vehicle_info(vehicle_raw):
    if not vehicle_raw:
        return {"make": None, "model": None, "year": None}

    return {
        "make": vehicle_raw.get("Make"),
        "model": vehicle_raw.get("Model"),
        "year": vehicle_raw.get("ModelYear")
    }

In [10]:
def enrich_contract_with_vehicle(row):
    vehicle_raw = get_vehicle_details(row["vin"])
    vehicle_info = extract_vehicle_info(vehicle_raw)

    return {
        "contract_id": row.get("contract_id") or row.get("id"),
        "sla": extract_sla_details(row),
        "vehicle": vehicle_info
    }

In [11]:
combined_records = []

for _, row in contracts_df.iterrows():
    combined_records.append(enrich_contract_with_vehicle(row))

combined_df = pd.DataFrame(combined_records)
combined_df.head()

,contract_id,sla,vehicle
0,1,"{'monthly_emi': 18500, 'interest_rate': 9.5, '...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."
1,2,"{'monthly_emi': 22000, 'interest_rate': 0.0, '...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."
2,3,"{'monthly_emi': 14500, 'interest_rate': 11.2, ...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."
3,4,"{'monthly_emi': 21000, 'interest_rate': 0.0, '...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."
4,5,"{'monthly_emi': 27500, 'interest_rate': 10.8, ...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."


In [12]:
print(contracts_df.columns.tolist())

['id', 'customer_name', 'contract_type', 'vehicle_type', 'monthly_emi', 'interest_rate', 'tenure_months', 'clause_summary', 'risk_flag', 'issue_type', 'recommended_action', 'vin']


In [ ]:
combined_df.to_csv("../data/milestone2_evaluation_output.csv", index=False)